In [ ]:
# ================================================================
# 21/9 HOLD-OUT VALIDATION — ONE CELL / ONE COMMAND
# ================================================================
# 30 nodes are divided reproducibly into:
#   21 calibration nodes
#    9 blind hold-out nodes
#
# IMPORTANT:
# The notebook calculates all metrics from the data. It does NOT
# force the values reported in the manuscript abstract.
#
# For a TRUE hold-out model validation, Q, D and lambda must first
# be calibrated using ONLY the 21 calibration nodes, and then fixed.
# The model must subsequently generate predictions for the 9
# hold-out nodes without recalibration.
#
# The present notebook uses the 30 observed/predicted pairs currently
# available in this project to demonstrate the complete 21/9
# evaluation workflow. Replace `predicted` with the predictions
# generated by the calibrated 2D-ADDS model for the final study.

# BOUNDARY-CONDITION CONSISTENCY NOTE
# ---------------------------------
# This notebook does NOT solve the FDM/PDE. It evaluates an existing set
# of observed and model-predicted values. Therefore, there is no boundary
# condition to modify in this notebook. The boundary condition used to
# generate the predictions must be defined in the forward-model notebook.
# The corresponding FDM notebook has been standardized to homogeneous
# Neumann boundaries (du/dn = 0) at inflow, lateral/open boundaries, and
# outflow.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ================================================================
# 1. THIRTY OBSERVED / RECONSTRUCTED VALUES
# ================================================================

observed = np.array([
    120, 145, 160, 180, 200,
    215, 230, 250, 270, 290,
    310, 330, 350, 370, 390,
    410, 430, 450, 470, 490,
    510, 530, 550, 570, 590,
    610, 630, 650, 670, 690
], dtype=float)

# ================================================================
# 2. THIRTY MODEL-PREDICTED VALUES
# ================================================================

predicted = np.array([
    118, 150, 158, 175, 205,
    220, 225, 248, 275, 295,
    305, 335, 345, 375, 392,
    405, 435, 448, 475, 485,
    515, 528, 548, 575, 585,
    615, 625, 655, 668, 695
], dtype=float)

if len(observed) != 30 or len(predicted) != 30:
    raise ValueError("Exactly 30 observed and 30 predicted values are required.")

if observed.shape != predicted.shape:
    raise ValueError("Observed and predicted arrays must have the same shape.")

# ================================================================
# 3. REPRODUCIBLE 21/9 RANDOM SPLIT
# ================================================================

rng = np.random.default_rng(42)

indices = np.arange(30)
rng.shuffle(indices)

calibration_idx = np.sort(indices[:21])
holdout_idx = np.sort(indices[21:])

observed_cal = observed[calibration_idx]
predicted_cal = predicted[calibration_idx]

observed_holdout = observed[holdout_idx]
predicted_holdout = predicted[holdout_idx]

# ================================================================
# 4. METRIC FUNCTION
# ================================================================

def calculate_metrics(observed_values, predicted_values):
    observed_values = np.asarray(observed_values, dtype=float)
    predicted_values = np.asarray(predicted_values, dtype=float)

    r2 = r2_score(observed_values, predicted_values)

    rmse = np.sqrt(
        np.mean((predicted_values - observed_values) ** 2)
    )

    mae = np.mean(
        np.abs(predicted_values - observed_values)
    )

    # MAPE: exclude zero observed values.
    nonzero = observed_values != 0

    mape = 100 * np.mean(
        np.abs(
            (predicted_values[nonzero] - observed_values[nonzero])
            / observed_values[nonzero]
        )
    )

    # FAC2: fraction satisfying 0.5 <= P/O <= 2.
    ratio = predicted_values[nonzero] / observed_values[nonzero]

    fac2 = np.mean(
        (ratio >= 0.5) & (ratio <= 2.0)
    )

    return r2, rmse, mae, mape, fac2

# ================================================================
# 5. CALIBRATION-SUBSET DIAGNOSTIC METRICS
# ================================================================

r2_cal, rmse_cal, mae_cal, mape_cal, fac2_cal = calculate_metrics(
    observed_cal,
    predicted_cal
)

# ================================================================
# 6. BLIND HOLD-OUT METRICS
# ================================================================

r2_val, rmse_val, mae_val, mape_val, fac2_val = calculate_metrics(
    observed_holdout,
    predicted_holdout
)

# ================================================================
# 7. DISPLAY THE 21/9 PARTITION AND METRICS
# ================================================================

print("=" * 70)
print("21/9 HOLD-OUT VALIDATION")
print("=" * 70)

print("\nCALIBRATION SUBSET")
print("-" * 70)
print(f"Number of calibration nodes: {len(calibration_idx)}")
print(f"Node numbers: {calibration_idx + 1}")

print("\nBLIND HOLD-OUT SUBSET")
print("-" * 70)
print(f"Number of hold-out nodes: {len(holdout_idx)}")
print(f"Node numbers: {holdout_idx + 1}")

print("\nCALIBRATION-SUBSET DIAGNOSTIC METRICS")
print("-" * 70)
print(f"R²   = {r2_cal:.4f}")
print(f"RMSE = {rmse_cal:.4f} particles")
print(f"MAE  = {mae_cal:.4f} particles")
print(f"MAPE = {mape_cal:.4f}%")
print(f"FAC2 = {fac2_cal:.4f}")

print("\nBLIND HOLD-OUT VALIDATION METRICS")
print("-" * 70)
print(f"R²   = {r2_val:.4f}")
print(f"RMSE = {rmse_val:.4f} particles")
print(f"MAE  = {mae_val:.4f} particles")
print(f"MAPE = {mape_val:.4f}%")
print(f"FAC2 = {fac2_val:.4f}")
print("=" * 70)

# ================================================================
# 8. DISPLAY THE 9 HOLD-OUT VALUES
# ================================================================

print("\n9 BLIND HOLD-OUT NODES")
print("-" * 70)
print(f"{'Node':>8} {'Observed':>14} {'Predicted':>14} {'Residual':>14}")
print("-" * 70)

for idx, O, P in zip(
    holdout_idx,
    observed_holdout,
    predicted_holdout
):
    print(
        f"{idx + 1:>8} "
        f"{O:>14.0f} "
        f"{P:>14.0f} "
        f"{O - P:>14.0f}"
    )

# ================================================================
# 9. FIGURE 1 — BLIND HOLD-OUT OBSERVED VS PREDICTED
# ================================================================

plt.figure(figsize=(7, 7))

plt.scatter(
    observed_holdout,
    predicted_holdout,
    s=100,
    alpha=0.8,
    label="Blind hold-out nodes"
)

xmin = min(observed_holdout.min(), predicted_holdout.min())
xmax = max(observed_holdout.max(), predicted_holdout.max())

plt.plot(
    [xmin, xmax],
    [xmin, xmax],
    linestyle="--",
    linewidth=2,
    label="1:1 Perfect Agreement"
)

metrics_text = (
    rf"$R^2 = {r2_val:.2f}$" "\n"
    rf"$RMSE = {rmse_val:.1f}$ particles" "\n"
    rf"$MAE = {mae_val:.1f}$ particles" "\n"
    rf"$MAPE = {mape_val:.2f}\%$" "\n"
    rf"$FAC2 = {fac2_val:.2f}$"
)

plt.text(
    0.05, 0.95, metrics_text,
    transform=plt.gca().transAxes,
    fontsize=12,
    verticalalignment="top",
    bbox=dict(boxstyle="round,pad=0.5", facecolor="white", alpha=0.85)
)

plt.xlabel("Observed Microplastic Counts (particles)", fontsize=13)
plt.ylabel("Model-Predicted Microplastic Counts (particles)", fontsize=13)
plt.title("Blind Hold-Out Evaluation: Observed vs. Model-Predicted Counts", fontsize=14)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

# ================================================================
# 10. FIGURE 2 — HOLD-OUT RESIDUALS
# ================================================================

residual_holdout = observed_holdout - predicted_holdout

plt.figure(figsize=(8, 5))

plt.axhline(
    0,
    linestyle="--",
    linewidth=1.5
)

plt.scatter(
    holdout_idx + 1,
    residual_holdout,
    s=90,
    alpha=0.8
)

plt.xlabel("Hold-Out Node", fontsize=13)
plt.ylabel("Residual (Observed − Predicted)", fontsize=13)
plt.title("Residuals for the 9 Blind Hold-Out Nodes", fontsize=14)
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

# ================================================================
# 11. INTERPRETATION
# ================================================================

print("\nINTERPRETATION")
print("-" * 70)
print(
    "The 21 nodes constitute the calibration subset and the 9 "
    "remaining nodes constitute the blind hold-out subset."
)
print(
    "The hold-out R², RMSE, MAE, MAPE and FAC2 above are calculated "
    "only from the 9 hold-out observed/predicted pairs."
)
print(
    "For the final manuscript, the 9 predicted values must come from "
    "the calibrated 2D-ADDS model after Q, D and lambda have been "
    "optimized using only the 21 calibration nodes."
)
print(
    "No metric has been forced to match the abstract."
)
print("=" * 70)
